### 1. What BIDS is, structurally
BIDS is a filesystem convention where information is split across two channels:
- <strong>filenames / directories</strong> encode structured entities like subject, session, task, run, suffix, extension
- <strong>JSON sidecars</strong> store metadata like `RepetitionTime`, `FlipAngle`, etc.
PyBIDS then indexes this structure so we can query it <strong>by entities</strong> instead of manually substring-matching filenames

### 2. What BIDSLayout is
```
layout = BIDSLayout("ds000228")
```
This creates an object representing the dataset rooted at `ds000228`. Internally, PyBIDS indexes the dataset and tracks files, entities, and associations so methods like `get()`, `get_file()`, and `get_entities()` can query them later. BIDSLayout is documented as a lightweight class representing a BIDS project file tree and supporting querying/manipulation of BIDS files.

### 3. What layout.get(...) is really doing
```
idx_files = layout.get(subject="pixar003", extension="nii.gz") # this is a list
```
tells the layout to filter by entity values and return all indexded files whose values satisfy those constraints. The returned items in the list are PyBIDS file objects, not plain strings. The docs describe BIDSFile as the basic file object, and also define subclasses like BIDSDataFile and BIDSJSONFile. These objects have attributes like `path`, `filename`, `dirname`, `entities`, and methods like `get_metadata()` and `get_associations()`.

### 4. What those attributes mean
When we print the attributes/methods (with the special methods filtered out) from one returned file object, the important ones are roughly:
- `path`: full path to the file
- `filename`: basename
- `dirname`: containing directory
- `entities`: parsed BIDS entities associated with that file
- `tags`: lower-level entity tagging machinery
- `metadata`: stored/linked metadata, though direct property access is discouraged in favor of getters
- `registry`, `class_`, etc.: more internal/model-ish bookkeeping

The two beginner workhorses are really just:
- `f.path`
- `f.entities`
- `f.get_metadata()`
Everything else is secondary until you need it.

### 5. The two most important methods on a returned file object
#### `f.get_entities()`
Returns the BIDS entities for that file. This is the structured parse of the filename and indexed information.

#### `f.get_metadata()`
Returns metadata associated with the file, typically assembled from sidecar JSONs using BIDS inheritance rules.
So the clean separation is:
> `entities` / `get_entities()` = filename-structured information<br>
> `get_metadata()` = JSON sidecar information and inherited metadata
That distinction is the whole game.

### 6. The minimal systematic workflow you should remember
When using PyBIDS, think in this order:
#### Step A: Query files
E.g.,
```
files = layout.get(subject="pixar003", extension=".nii.gz", return_type="object")
```
#### Step B: Inspect one file
```
f = files[0]
print(type(f))
print(f.path)
print(f.entities)
print(f.get_metadata())
```
#### Step C: Decide whether your target lives in entities or metadata
subject / task / run / suffix / extension → entities
FlipAngle / TR / EchoTime → metadata

This division is grounded in the PyBIDS/BIDS design itself.

In [1]:
import os 

PROJECT_FOLDER_PATH = "/Users/jowanglin/brainhack-ntu/mne-bids"
DS000228 = "ds000228"

def get_main_folders(data_folder: str, project_folder_path: str = None):
    if project_folder_path is None:
        project_folder_path = "~"
    walk = os.walk(f"{project_folder_path}/{data_folder}")
    main_folders, got_subj_folder = {}, 0

    for parent, _, files in walk:
        if parent == f"{PROJECT_FOLDER_PATH}/{data_folder}":
            top_level_files = [f for f in files]
        else:
            if parent.split("/")[-1][:3] == "sub": 
                got_subj_folder += 1
                if got_subj_folder >=2:
                    continue 
            elif "sub" in parent and got_subj_folder >= 2:
                continue 
            parent = parent.replace(f"{PROJECT_FOLDER_PATH}/{data_folder}", "..")
            main_folders[parent] = [f for f in files]

    return top_level_files, main_folders

top_level_files, main_folders = get_main_folders(DS000228, PROJECT_FOLDER_PATH)

print(f"**MAIN FOLDERS & FILES OF THE DS005509 DATASET**\n\n{DS000228}")
for top_lv_f in top_level_files:
    print(f"""  |___ {top_lv_f}""")

print("\n")

subject_folders = {k: v for k, v in main_folders.items() if "sub" in k}
for subdir, files in subject_folders.items():
    if not files:
        print(f"  |__ {subdir[3:]}")
    else:
        print(f"      |__ {subdir.split('/')[-1]}")
        print(f"          {files}")

print("\n")

other_folders = {k: v for k, v in main_folders.items() if "sub" not in k and k.split("/")[1][0] != "."}
for subdir, files in other_folders.items():
    if len(subdir[3:].split("/")) == 1:
        print(f"  |__ {subdir[3:]}")
        print(f"        {files}")
    elif len(subdir[3:].split("/")) == 2:
        print(f"      |__ {subdir.split('/')[-1]}")
        if len(files) > 5:
            print(f"            {files[:5]}...")
            file_extensions = set([f.split(".")[-1] for f in files])
            print(f"            >> extensions = {file_extensions}")          
        else:
            print(f"            {files}")
    else:
        print(f"          |__ {subdir.split('/')[-1]}")
        if len(files) > 5:
            print(f"                 {files[:5]}...")
            file_extensions = set([f.split(".")[-1] for f in files])
            print(f"                  >> extensions = {file_extensions}")  
        else:
            print(f"                 {files}")


**MAIN FOLDERS & FILES OF THE DS005509 DATASET**

ds000228
  |___ task-pixar_bold.json
  |___ CHANGES
  |___ README
  |___ dataset_description.json
  |___ .gitattributes
  |___ participants.json
  |___ participants.tsv


  |__ sub-pixar058
      |__ anat
          ['sub-pixar058_T1w.json', 'sub-pixar058_T1w.nii.gz']
      |__ func
          ['sub-pixar058_task-pixar_bold.nii.gz', 'sub-pixar058_task-pixar_bold.json']


  |__ derivatives
        ['README.txt']
      |__ preprocessed_data
            []
      |__ ROIs
            ['right_primotor.mat', 'lSTS.mat', 'RMFG_9mm_sphere_xyz.mat', 'LInsula_9mm_sphere_xyz.mat', 'lRSC.mat']...
            >> extensions = {'mat'}
      |__ mriqc
            ['T1w_group.html', 'bold_group.html', 'bold.csv', 'T1w.csv']
          |__ derivatives
                 ['sub-pixar001_T1w.json', 'sub-pixar112_task-pixar_bold.json', 'sub-pixar033_task-pixar_bold.json', 'sub-pixar109_T1w.json', 'sub-pixar108_T1w.json']...
                  >> extensions = {'jso

In [ ]:
# I need npx because my bids-validator's scope is installed locally inside my project folder
# not globally (system-wide)

!npx bids-validator "{PROJECT_FOLDER_PATH}/{DS000228}"

⠙bids-validator@1.15.0
(node:44827) Warning: Closing directory handle on garbage collection
(Use `node --trace-warnings ...` to show where the warning was created)
	1: [ERR] A json sidecar file was found without a corresponding data file (code: 90 - SIDECAR_WITHOUT_DATAFILE)
		./sub-pixar004/anat/sub-pixar004_T1w.json
		./sub-pixar004/func/sub-pixar004_task-pixar_bold.json
		./sub-pixar005/anat/sub-pixar005_T1w.json
		./sub-pixar005/func/sub-pixar005_task-pixar_bold.json
		./sub-pixar006/anat/sub-pixar006_T1w.json
		./sub-pixar006/func/sub-pixar006_task-pixar_bold.json
		./sub-pixar007/anat/sub-pixar007_T1w.json
		./sub-pixar007/func/sub-pixar007_task-pixar_bold.json
		./sub-pixar008/anat/sub-pixar008_T1w.json
		./sub-pixar008/func/sub-pixar008_task-pixar_bold.json
		... and 294 more files having this issue (Use --verbose to see them all).

	Please visit https://neurostars.org/search?q=SIDECAR_WITHOUT_DATAFILE for existing conversations about this issue.

	1: [WARN] Task scans should h

In [158]:
os.chdir(f"{PROJECT_FOLDER_PATH}/{DS000228}")
!ls -l sub-pixar004/anat
print("\n")
!test -L sub-pixar004/anat/sub-pixar004_T1w.nii.gz && echo SYMLINK || echo NOT_SYMLINK
!test -e sub-pixar004/anat/sub-pixar004_T1w.nii.gz && echo REAL || echo MISSING

total 296
-rw-r--r--@ 1 jowanglin  staff  150118 Mar 26 19:24 sub-pixar004_T1w.json
lrwxr-xr-x@ 1 jowanglin  staff     140 Mar 26 19:24 sub-pixar004_T1w.nii.gz -> ../../.git/annex/objects/K7/05/MD5E-s5905926--8ed164d4ef7f18dbf9d1414d7e152e5d.nii.gz/MD5E-s5905926--8ed164d4ef7f18dbf9d1414d7e152e5d.nii.gz


SYMLINK
MISSING


<strong>SYMLINK</strong> $\land$ <strong>MISSING</strong> $\implies$ broken symlink.
> The .nii.gz paths are probably annex symlinks pointing into git-annex object storage, and those object files are missing because the actual content has not been fetched yet (I haven't `datalad get` those subjects, only 1, 2, 3).


In [2]:
from bids import BIDSLayout
from bids.layout.models import BIDSFile

# BIDSLayout is the central PyBIDS class representing an indexed BIDS dataset
# .get() is its main query method for retrieving matching files or metadata

layout = BIDSLayout(f"{PROJECT_FOLDER_PATH}/{DS000228}")
print(f"type of `layout` = {type(layout)}")
idx_files = layout.get(subject="pixar003", extension=".nii.gz", return_type="object") # extensions should start with a leading dot
print(f">> Grabbed {len(idx_files)} indexed files from queried entites\n")

file = idx_files[0]
print(f"{str(type(file)).split('.')[-1][:-2]} is a subclass of BIDSFile:  {issubclass(type(file), BIDSFile)}\n")
print(f"{type(file)} has attributes and methods:")

for attr in dir(file):
    if attr[0] != "_":
        print("  ", attr)

type of `layout` = <class 'bids.layout.layout.BIDSLayout'>
>> Grabbed 2 indexed files from queried entites

BIDSImageFile is a subclass of BIDSFile:  True

<class 'bids.layout.models.BIDSImageFile'> has attributes and methods:
   class_
   copy
   dirname
   entities
   filename
   get_associations
   get_entities
   get_image
   get_metadata
   is_dir
   metadata
   path
   registry
   relpath
   tags


In [ ]:
bolds = layout.get(subject="pixar003", suffix="bold", extension=".nii.gz", return_type="object")
print(f">> Grabbed {len(bolds)} indexed files from queried entites\n")

bold = bolds[0]
print(f"{bold.dirname.replace(PROJECT_FOLDER_PATH, '')}")
print(f"    |__{bold.filename}")
print(bold.entities)
print(bold.get_entities(metadata=False, values="objects"))
print(bold.metadata)
print(bold.get_metadata())
print(bold.get_associations())

In [4]:
layout_df = layout.to_df()
layout_df["path"] = layout_df["path"].apply(lambda s: s.replace(PROJECT_FOLDER_PATH, ".."))
display(layout_df)

entity,path,datatype,extension,subject,suffix,task
0,../ds000228/dataset_description.json,NaN,.json,NaN,description,NaN
1,../ds000228/derivatives/README.txt,NaN,.txt,NaN,README,NaN
2,../ds000228/participants.json,NaN,.json,NaN,participants,NaN
3,../ds000228/participants.tsv,NaN,.tsv,NaN,participants,NaN
4,../ds000228/sub-pixar001/anat/sub-pixar001_T1w...,anat,.json,pixar001,T1w,NaN
...,...,...,...,...,...,...
622,../ds000228/sub-pixar155/func/sub-pixar155_tas...,func,.json,pixar155,bold,pixar
623,../ds000228/sub-pixar155/func/sub-pixar155_tas...,func,.nii.gz,pixar155,bold,pixar
624,../ds000228/task-pixar_bold.json,NaN,.json,NaN,bold,pixar
625,../ds000228/CHANGES,NaN,NaN,NaN,NaN,NaN


In [5]:
niftis = layout.get(extension=".nii.gz", return_type="object")
unique_flip_angles = set([f.get_metadata()["FlipAngle"] for f in niftis])
print(f"Flip angles (unique set) = {unique_flip_angles}")

subjects = layout.get_subjects()
flip_angles_per_subject = {sub: [f.get_metadata()["FlipAngle"]
                                   for f in layout.get(subject=sub, extension=".nii.gz", return_type="object")]
                                   for sub in subjects}
not_7or90 = {k: v for k, v in flip_angles_per_subject.items() if v != [7, 90]}
print(f"\nSubjects whose flip angles aren't [7, 90]:\n   {not_7or90}")


Flip angles (unique set) = {7, 84, 88, 89, 90}

Subjects whose flip angles aren't [7, 90]:
   {'pixar012': [7, 89], 'pixar019': [7, 84], 'pixar028': [7, 88]}


In [7]:
import pandas as pd

info_tsv = pd.read_csv(f"{PROJECT_FOLDER_PATH}/ds000224/participants.tsv", sep = "\t")
info_tsv

,participant_id,gender,age,education_degree,education_years
0,sub-MSC01,M,34,Doctorate,22.0
1,sub-MSC02,M,34,Doctorate,28.0
2,sub-MSC03,F,29,Masters,18.0
3,sub-MSC04,F,28,Bachelors,22.0
4,sub-MSC05,M,27,Bachelors,20.0
5,sub-MSC06,F,24,Bachelors,17.5
6,sub-MSC07,F,31,Masters,20.0
7,sub-MSC08,F,27,Professional,21.0
8,sub-MSC09,M,26,Professional,19.0
9,sub-MSC10,M,31,Professional,19.0


### HBN-EEG Dataset
More complex
> I downloaded it usinf OpenNeuro CLI.

In [8]:
from bids import BIDSLayout
from bids.layout.models import BIDSFile
import pandas as pd
import os
from IPython.display import clear_output
from time import perf_counter

PROJECT_FOLDER_PATH = "/Users/jowanglin/brainhack-ntu/mne-bids"
HBN_EEG = "ds005509"
subjects = ["NDAREC480KFA"]

if HBN_EEG != os.getcwd().split("/")[-1]:
    try:
        os.chdir(f"{PROJECT_FOLDER_PATH}/{HBN_EEG}")
    except FileNotFoundError:
        raise Exception ("Data folder not found...")
    
START = perf_counter()
!datalad get sub-NDARVG761NR2
END = perf_counter()
clear_output()

DELTA = END - START
MIN, SEC = int((DELTA % 3600) // 60), DELTA % 60
print(f"Downloaded data for {len(subjects)} subjects: {subjects}\n... Took {MIN}m {SEC:.2f}s")


Downloaded data for 1 subjects: ['NDAREC480KFA']
... Took 0m 1.42s


In [9]:
top_level_files, main_folders = get_main_folders(HBN_EEG, PROJECT_FOLDER_PATH)

print(f"**MAIN FOLDERS & FILES OF THE DS005509 DATASET**\n\n{HBN_EEG}")
for top_lv_f in top_level_files:
    print(f"""  |___ {top_lv_f}""")

print("\n")

for subdir, files in main_folders.items():
    if subdir.split("/")[1][0] != ".":
        file_extensions = set([f.split(".")[-1] for f in files])
        if file_extensions and "/" in subdir[3:]:
            print(f"  |___ {subdir[3:].split('/')[0]}")           
            print(f"         |___ {subdir[3:].split('/')[1]}")
            print(f"              {file_extensions}")
        elif file_extensions: 
            print(f"  |___ {subdir[3:]}")
            print(f"       {file_extensions}")
    

**MAIN FOLDERS & FILES OF THE DS005509 DATASET**

ds005509
  |___ task-DespicableMe_events.json
  |___ task-ThePresent_eeg.json
  |___ task-seqLearning6target_eeg.json
  |___ CHANGES
  |___ task-seqLearning8target_events.json
  |___ task-ThePresent_events.json
  |___ task-FunwithFractals_eeg.json
  |___ task-seqLearning8target_eeg.json
  |___ task-surroundSupp_events.json
  |___ README
  |___ task-DiaryOfAWimpyKid_events.json
  |___ task-contrastChangeDetection_events.json
  |___ task-symbolSearch_eeg.json
  |___ task-RestingState_eeg.json
  |___ task-FunwithFractals_events.json
  |___ task-RestingState_events.json
  |___ task-DiaryOfAWimpyKid_eeg.json
  |___ dataset_description.json
  |___ task-contrastChangeDetection_eeg.json
  |___ .gitattributes
  |___ task-DespicableMe_eeg.json
  |___ task-symbolSearch_events.json
  |___ participants.json
  |___ participants.tsv
  |___ task-surroundSupp_eeg.json
  |___ task-seqLearning6target_events.json


  |___ sub-NDARHT019ER6
         |___ eeg

In [10]:
os.chdir(PROJECT_FOLDER_PATH)

layout = BIDSLayout(HBN_EEG)
print(f"type of `layout` = {type(layout)}")
idx_files = layout.get(subject=subjects[0], extension="set", return_type="object")  # set files are eeg data files
print(f">> Grabbed {len(idx_files)} indexed files from queried entites\n")

file = idx_files[0]
print(f"{str(type(file)).split('.')[-1][:-2]} is a subclass of BIDSFile:  {issubclass(type(file), BIDSFile)}\n")
print(f"{type(file)} has attributes and methods:")

for attr in dir(file):
    if attr[0] != "_": # and attr[-1] != "_":
        print("  ", attr)

type of `layout` = <class 'bids.layout.layout.BIDSLayout'>
>> Grabbed 12 indexed files from queried entites

BIDSFile is a subclass of BIDSFile:  True

<class 'bids.layout.models.BIDSFile'> has attributes and methods:
   class_
   copy
   dirname
   entities
   filename
   get_associations
   get_entities
   get_metadata
   is_dir
   metadata
   path
   registry
   relpath
   tags


In [121]:
print(file.dirname)
print(file.filename)
print(file.class_)
print(file.entities)
print(file.get_entities())
print(file.metadata)
print(file.get_metadata())

/Users/jowanglin/brainhack-ntu/mne-bids/ds005509/sub-NDARVG761NR2/eeg
sub-NDARVG761NR2_task-contrastChangeDetection_run-2_eeg.set
file
{'EEGChannelCount': 129, 'EEGReference': 'Cz', 'InstitutionAddress': '101 E 56th St, New York, NY 10022', 'InstitutionName': 'Child Mind Institute', 'Instructions': 'Fixate on the central dot. Press the LEFT button with LEFT hand when the LEFT-tilted pattern gets stronger. Press the RIGHT button with RIGHT hand when the RIGHT-tilted pattern gets stronger. Work as quickly as you can without making mistakes. Press the mouse button to begin.', 'Manufacturer': 'Magtism EGI', 'ManufacturersModelName': '128-channel GSN 200 v.2.1', 'PowerLineFrequency': 60, 'RecordingDuration': 302.842, 'RecordingType': 'continuous', 'SamplingFrequency': 500, 'SoftwareFilters': 'n/a', 'TaskDescription': "The contrast change detection paradigm is designed to enable isolation of the neural signatures of sensory evidence encoding, accumulation, and motor preparation without the n